<img src= "../images/recommendation/agricultural_feed_banner.png" width = "800">
</img>


<h2 style= "color: #78C34D; text-align: left; font-weight: bold;">
Agricultural Feed Recommendation Engine
</h2>

<h3 style= "color: #CDAA2B; text-align: left; font-weight: bold;">
Business Understanding
</h3>

An agricultural supply company wants to recommend similar feed types to farmers based on feed performance data

The objective is to use the Chickwts dataset as a proxy for agricultural feed performance and build a recommendation system that identifies comparable feed types. PCA will be used for dimensionality reduction, while similarity metrics will be used to determine which feeds have similar performance characteristics

The final system will recommend feed types that may produce similar results

<h3 style= "color: #CDAA2B; text-align: left; font-weight: bold;">
1. Data Understanding
</h3>

In [1]:
# Import of required libs
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# load in the dataset
chick_df = pd.read_csv("../data/chickwts.csv")

# display first 5 rows
chick_df.head()

,weight,feed
0,179,horsebean
1,160,horsebean
2,136,horsebean
3,227,horsebean
4,217,horsebean


In [2]:
chick_df.shape

(71, 2)

In [3]:
chick_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 71 entries, 0 to 70
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   weight  71 non-null     int64 
 1   feed    71 non-null     object
dtypes: int64(1), object(1)
memory usage: 1.2+ KB


In [4]:
chick_df.isnull().sum()

weight    0
feed      0
dtype: int64

In [5]:
chick_df.duplicated().sum()

1

In [7]:
# investigate the duplicate
chick_df[chick_df.duplicated(keep=False)]

,weight,feed
24,248,soybean
35,248,soybean


In [6]:
chick_df["feed"].value_counts()

feed
soybean      14
linseed      12
sunflower    12
casein       12
meatmeal     11
horsebean    10
Name: count, dtype: int64

<h4 style= "color: #CDAA2B; text-align: left; font-weight: bold;">
Data Understanding Findings
</h4>

The dataset contains 71 observations and two variables: `weight`, representing chicken weight, and `feed`, representing the type of feed used.

The dataset contains no missing values. One pair of identical observations was identified, where two soybean-fed chickens have a recorded weight of 248.

Since the dataset does not contain a unique identifier, it cannot be determined whether this represents a duplicate record or two different chickens with the same weight and feed. Therefore, the observations were retained to avoid removing potentially valid data.

The dataset also contains six different feed types with relatively similar numbers of observations.

Since PCA cannot meaningfully operate directly on just weight and the categorical feed column

We'll first summarize each feed's performance:

In [8]:
feed_summary = chick_df.groupby("feed")["weight"].agg(
    mean_weight="mean",
    median_weight="median",
    std_weight="std",
    min_weight="min",
    max_weight="max"
).round(2)

feed_summary

,mean_weight,median_weight,std_weight,min_weight,max_weight
feed,,,,,
casein,323.58,342.0,64.43,216,404
horsebean,160.20,151.5,38.63,108,227
linseed,218.75,221.0,52.24,141,309
meatmeal,276.91,263.0,64.90,153,380
soybean,246.43,248.0,54.13,158,329
sunflower,328.92,328.0,48.84,226,423


<h3 style= "color: #CDAA2B; text-align: left; font-weight: bold;">
2. Standardizing and Applying PCA
</h3>

In [9]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# we now transform each column so its measured equally on a comparable scale
scaler = StandardScaler()
feed_scaled = scaler.fit_transform(feed_summary)

# we apply PCA, our 5 xtics are hard to visualize, pca creates new variables(2 prnciple components)
pca = PCA(n_components=2)
feed_pca = pca.fit_transform(feed_scaled)

# check how much info is retained
print("Explained variance:", pca.explained_variance_ratio_)
print("Total explained variance:", pca.explained_variance_ratio_.sum())

Explained variance: [0.8676257  0.12214042]
Total explained variance: 0.9897661189299085


In [10]:
# calculate feed similarity
from sklearn.metrics.pairwise import cosine_similarity

similarity_matrix = cosine_similarity(feed_pca)

similarity_df = pd.DataFrame(
    similarity_matrix,
    index=feed_summary.index,
    columns=feed_summary.index
)

similarity_df.round(2)

feed,casein,horsebean,linseed,meatmeal,soybean,sunflower
feed,,,,,,
casein,1.00,-1.00,-0.95,0.59,-0.87,0.78
horsebean,-1.00,1.00,0.94,-0.61,0.86,-0.76
linseed,-0.95,0.94,1.00,-0.31,0.98,-0.93
meatmeal,0.59,-0.61,-0.31,1.00,-0.12,-0.05
soybean,-0.87,0.86,0.98,-0.12,1.00,-0.98
sunflower,0.78,-0.76,-0.93,-0.05,-0.98,1.00


In [11]:
# build the recommender function
def recommend_feeds(feed_name, n=2):
    similarities = similarity_df[feed_name].drop(feed_name)
    return similarities.sort_values(ascending=False).head(n)

In [12]:
# test it
recommend_feeds("soybean")

feed
linseed      0.981276
horsebean    0.859742
Name: soybean, dtype: float64

In [13]:
recommend_feeds("casein")

feed
sunflower    0.775139
meatmeal     0.589345
Name: casein, dtype: float64

<h4 style= "color: #CDAA2B; text-align: left; font-weight: bold;">
Results
</h4>

The first two principal components retained approximately 98.98% of the variance in the feed performance characteristics, indicating that the two-dimensional representation preserves most of the information from the original performance measures.

Using cosine similarity, sunflower was identified as the feed most similar to casein, with a similarity score of approximately 0.78, followed by meatmeal at approximately 0.59. This suggests that sunflower has the most comparable performance profile to casein among the available feed types.


<h4 style= "color: #CDAA2B; text-align: left; font-weight: bold;">
Conclusion
</h4>

The recommendation system successfully combines PCA and cosine similarity to identify feeds with comparable performance profiles. PCA reduced the performance characteristics to two components while retaining approximately 98.98% of the variance, and the similarity-based function can recommend the closest feed alternatives for a selected feed.

Since the dataset contains only six feed types, the system should be considered a prototype rather than a production recommendation engine.